In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
from alphagenome_pytorch.plotting.splicing import (
    plot_splice_site_dynamics,
    plot_splice_site_predictions,
    plot_usage_density,
    plot_splice_site_usage_by_tissue,
    plot_splice_site_dynamics_from_usage_results,
    SPLICE_CLASS_NAMES,
    BACKGROUND_CLASS,
    CLASS_LABELS,
    CLASS_COLORS,
    TISSUE_COLORS,
    TISSUE_ORDER,
    SPECIES_ORDER,
    SPECIES,
    SPECIES_SCI,
    CHR_SIZES,
)
from alphagenome_pytorch.evaluation.splicing import classify_sites_by_true_usage_variance

In [ ]:
work_dir = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/"
ann_data_dir = "/home/elek/sds/sd17d003/Anamaria/alphagenome_genomicsxai/data/"
model = "ft_human_mouse_rat_rabbit_opossum"
model_dir = os.path.join(work_dir, model)

subsets = ["intersect_protein_coding"] # "intersect_usage", "intersect_protein_coding", "gtf_protein_coding",
subset = subsets[0]
pred_dir = f"preds_{subset}"
conf_fn = f"data_config_{subset}.json"
data_config_path = os.path.join(ann_data_dir, conf_fn)
data_config = json.load(open(data_config_path, "r"))

## Load splice usage predictions

Reload the raw per-site usage predictions (chr_pos / tissue / timepoint / true / pred
arrays), split out of `splice_model_eval.ipynb`'s "Parse splice usage predictions"
section, which the trajectory and developmental-dynamics analyses below need.

In [ ]:
%%time
usage_results = {}

for sps in SPECIES_ORDER:
    usage_results[sps] = {}

    # Load condition metadata: maps condition index -> "Tissue_Timepoint" label
    metadata_file = os.path.join(ann_data_dir, SPECIES_SCI[sps], "usage.json")
    if not os.path.exists(metadata_file):
        print(f"Metadata missing for {sps}: {metadata_file}")
        continue
    with open(metadata_file) as f:
        metadata = json.load(f)
    idx_to_label = {int(v): k for k, v in metadata.get("condition_labels", {}).items()}

    if pred_dir.startswith("preds"):
        pd_key = pred_dir
    else:
        pd_key = "preds_" + pred_dir
    usage_file = os.path.join(work_dir, model, f"{pd_key}", sps, f"usage_{sps}.npz")
    if not os.path.exists(usage_file):
        print(f"  Usage file missing: {sps} / {pd_key}")
        usage_results[sps][pd_key] = None
        continue

    data = np.load(usage_file)
    chr_pos    = data["chr_pos"]
    cond_ids   = data["cond_ids"]
    trues      = data["true"]
    preds      = data["pred"]

    # Split condition labels into Tissue and Timepoint
    tissues = []        
    timepoints = []
    for cond_id in cond_ids:
        tissue, timepoint = idx_to_label[cond_id].split("_")
        tissues.append(tissue)
        timepoints.append(timepoint)

    usage_results[sps] = {
        "chr_pos": chr_pos,
        "cond_ids": cond_ids,
        "tissues": tissues,
        "timepoints": timepoints,
        "trues": trues,
        "preds": preds,
        "idx_to_label": idx_to_label
    }

### Calculate trajectory correlations

In [ ]:
# Collect usage observations for all available species
usage_df_by_species = {}
missing_species = []

for sp in SPECIES_ORDER:
    sp_data = usage_results.get(sp)

    # keep only species with direct arrays loaded in usage_results
    if not isinstance(sp_data, dict) or not {"chr_pos", "trues", "preds", "cond_ids"}.issubset(sp_data.keys()):
        missing_species.append(sp)
        continue

    usage_df_by_species[sp] = pd.DataFrame({
        "species": sp,
        "chr_pos": np.asarray(sp_data["chr_pos"]).astype(str),
        "true_usage": np.asarray(sp_data["trues"], dtype=float),
        "pred_usage": np.asarray(sp_data["preds"], dtype=float),
        "cond_idx": np.asarray(sp_data["cond_ids"], dtype=int),
        'tissue': np.asarray(sp_data["tissues"], dtype=str),
        'timepoint': np.asarray(sp_data["timepoints"], dtype=str),
    })

if missing_species:
    print(f"Skipped species with missing usage arrays: {missing_species}")

usage_df_all = pd.concat(usage_df_by_species.values(), ignore_index=True)
print(f"Collected {len(usage_df_all):,} rows from {len(usage_df_by_species)} species.")
usage_df_all

In [ ]:
%%time

def _pearson_safe(g):
    x = g["true_usage"].to_numpy(dtype=float)
    y = g["pred_usage"].to_numpy(dtype=float)
    if len(x) < 2 or np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def _mean_safe(s):
    arr = s.to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    return np.mean(arr)

site_corr_all = (
    usage_df_all.groupby(["species", "chr_pos", "tissue"], as_index=False)
    .apply(lambda g: pd.Series({
        "pearson_r": _pearson_safe(g),
        "mae": np.sqrt(np.mean((g["true_usage"] - g["pred_usage"]) ** 2)),
        "cosine": (
            np.dot(g["true_usage"], g["pred_usage"]) /
            (np.linalg.norm(g["true_usage"]) * np.linalg.norm(g["pred_usage"]))
            if np.linalg.norm(g["true_usage"]) > 0 and np.linalg.norm(g["pred_usage"]) > 0
            else np.nan
        ),
        "n_obs_usage": int(g.loc[g["true_usage"] > 0, "cond_idx"].nunique()),
        "mean_obs_usage": _mean_safe(g.loc[g["true_usage"] > 0, "true_usage"]),
        "mean_true_usage": _mean_safe(g["true_usage"]),
        "mean_pred_usage": _mean_safe(g["pred_usage"]),
        "min_obs_usage": g.loc[g["true_usage"] > 0, "true_usage"].min(),
        "min_true_usage": _mean_safe(g["true_usage"]),
        "max_obs_usage": g["true_usage"].max(),
    }))
    .reset_index(drop=True)
    .sort_values(["species", "pearson_r"], ascending=[True, False])
    .reset_index(drop=True)
)

# How many sites have no observations of true usage > 0 i.e. min_obs is NaN?
n_no_obs = site_corr_all["min_obs_usage"].isna().sum()
print(f"Number of sites with no observations of true usage > 0: {n_no_obs} / {len(site_corr_all)} ({n_no_obs / len(site_corr_all):.2%})")

# Remove sites with no observations of true usage > 0
site_corr_all = site_corr_all[~site_corr_all["min_obs_usage"].isna()].reset_index(drop=True)
site_corr_all

### Trajectories across all sites

In [ ]:
species_list = sorted(site_corr_all['species'].unique())
n_species = len(species_list)

fig, axes = plt.subplots(
    n_species, 2,
    figsize=(10, n_species * 2.5),
    sharey=False
)

if n_species == 1:
    axes = axes.reshape(1, -1)

for row, species in enumerate(species_list):
    sp_data = site_corr_all[(site_corr_all['species'] == species) & (site_corr_all['n_obs_usage'] > 3) ].copy()
    
    # Pearson r distribution
    pearson_data = sp_data["pearson_r"].dropna()
    mean_r = pearson_data.mean()
    median_r = pearson_data.median()
    sns.histplot(pearson_data, bins=50, kde=True, color="#4C72B0", ax=axes[row, 0])
    axes[row, 0].axvline(mean_r, color="#d62728", linewidth=1.5, linestyle="-", label=f"Mean={mean_r:.3f}")
    axes[row, 0].axvline(median_r, color="#2ca02c", linewidth=1.5, linestyle="-", label=f"Median={median_r:.3f}")
    axes[row, 0].set_xlim(-1.1, 1.1)
    axes[row, 0].set_ylabel(species, fontsize=10)
    axes[row, 0].set_xlabel("Pearson r")
    axes[row, 0].legend(frameon=False, fontsize=8)
    axes[row, 0].grid(alpha=0.3)

    # MAE distribution
    mae_data = sp_data["mae"].dropna()
    mean_mae = mae_data.mean()
    median_mae = mae_data.median()
    sns.histplot(mae_data, bins=50, kde=True, color="#ff7f0e", ax=axes[row, 1])
    axes[row, 1].axvline(mean_mae, color="#d62728", linewidth=1.5, linestyle="-", label=f"Mean={mean_mae:.4f}")
    axes[row, 1].axvline(median_mae, color="#2ca02c", linewidth=1.5, linestyle="-", label=f"Median={median_mae:.4f}")
    axes[row, 1].set_xlim(-0.05, 0.95)
    axes[row, 1].set_xlabel("MAE")
    axes[row, 1].legend(frameon=False, fontsize=8)
    axes[row, 1].grid(alpha=0.3)

plt.suptitle(f"Trajectory Evaluation: {model} / {pd_key}", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(work_dir, model, f"{pd_key}", "trajectory_correlation_distributions.png"), dpi=300)

Correlation is the lowest for the sites with low true usage. These could be the sites with overall low usage, or sites with usage observed in only a small number of conditions.

In [ ]:
species_plot = sorted(site_corr_all["species"].dropna().unique())
metric_specs = [
    ("pearson_r", "Pearson r", "#1f77b4", (-1.05, 1.05)),
    ("mae", "MAE", "#ff7f0e", (-0.05, 0.95)),
]
n_bins = 20
fig, axes = plt.subplots(
    len(species_plot), len(metric_specs),
    figsize=(12, max(2.5 * len(species_plot), 3.0)),
    sharex=False,
    squeeze=False
)
for r, sp in enumerate(species_plot):
    sp_df = site_corr_all[site_corr_all["species"] == sp].copy()
    for c, (metric_key, metric_label, color, ylim) in enumerate(metric_specs):
        ax = axes[r, c]
        dfm = sp_df[np.isfinite(sp_df["n_obs_usage"]) & np.isfinite(sp_df[metric_key])].copy()

        if len(dfm) > 0 and dfm["n_obs_usage"].nunique() > 1:
            q = min(n_bins, dfm["n_obs_usage"].nunique())
            if q >= 2:
                dfm["n_obs_bin"] = pd.qcut(dfm["n_obs_usage"], q=q, duplicates="drop")

                # Use bin midpoint as numeric x position
                dfm["bin_mid"] = dfm["n_obs_bin"].apply(lambda b: (b.left + b.right) / 2)
                bin_order = sorted(dfm["bin_mid"].unique())

                box_data = [
                    dfm.loc[dfm["bin_mid"] == mid, metric_key].dropna().values
                    for mid in bin_order
                ]
                box_data = [v for v in box_data if len(v) > 0]
                positions = [mid for mid, v in zip(bin_order, box_data) if len(v) > 0]

                if box_data:
                    # Scale box width to ~80% of the smallest gap between positions
                    gaps = np.diff(sorted(positions))
                    width = 0.8 * gaps.min() if len(gaps) > 0 else 1.0

                    bp = ax.boxplot(
                        box_data,
                        positions=positions,
                        widths=width,
                        patch_artist=True,
                        showfliers=False,
                        manage_ticks=False
                    )
                    for patch in bp["boxes"]:
                        patch.set_facecolor(color)
                        patch.set_alpha(0.6)
                        patch.set_edgecolor(color)
                    for element in ["whiskers", "caps"]:
                        for artist in bp[element]:
                            artist.set_color(color)
                            artist.set_linewidth(0.8)
                    for median in bp["medians"]:
                        median.set_color("#333333")
                        median.set_linewidth(1.5)

                    x_min, x_max = dfm["n_obs_usage"].min(), dfm["n_obs_usage"].max()
                    pad = max(1.0, 0.02 * (x_max - x_min))
                    ax.set_xlim(x_min - pad, x_max + pad)

        ax.set_ylim(*ylim)
        ax.grid(alpha=0.25)
        ax.set_xlabel("Number of observed timepoints")
        # Integer x labels with step to avoid overcrowding
        step = max(1, len(positions) // 10)  # show ~10 labels max
        ax.set_xticks(positions)
        ax.set_xticklabels(
            [str(int(round(p))) if i % step == 0 else "" for i, p in enumerate(positions)], fontsize=8
        )
        ax.set_ylabel(metric_label)
        if r == 0:
            ax.set_title(metric_label)
        if c == 0:
            ax.text(
                -0.25, 0.5, sp,
                transform=ax.transAxes,
                rotation=90,
                va="center", ha="center", fontsize=10
            )

plt.suptitle(f"Trajectory Evaluation: {model} / {pd_key}", fontsize=12)
plt.tight_layout()
plt.savefig(
    os.path.join(work_dir, model, f"{pd_key}", "trajectory_correlation_vs_number_observed_sites.png"),
    dpi=300, bbox_inches="tight"
)

In [ ]:
# Scatterplot of MAE pearson correlation; one plot for every species

species_plot = sorted(site_corr_all["species"].dropna().unique())
n_species = len(species_plot)

fig, axes = plt.subplots(1, n_species, figsize=(n_species * 4, 4), sharey=True)

if n_species == 1:
    axes = [axes]

for ax, sp in zip(axes, species_plot):
    sp_data = site_corr_all[site_corr_all["species"] == sp].copy()
    
    # Filter for finite values
    sp_data_clean = sp_data[np.isfinite(sp_data["mae"]) & np.isfinite(sp_data["pearson_r"])].copy()
    
    scatter = ax.scatter(
        sp_data_clean["mae"],
        sp_data_clean["pearson_r"],
        s=1,
        alpha=0.3,
        color="#1f77b4",
        edgecolors="none"
    )
    
    ax.set_xlabel("MAE", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Pearson r", fontsize=10)
    ax.set_title(sp, fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.show()
fig.savefig(
    os.path.join(work_dir, model, f"{pd_key}", "trajectory_mae_vs_pearson.png"),
    dpi=300, bbox_inches="tight"
)

### Splice site examples

Function to plot splice site trajectories for selected sites

 Extract condition mapping from the data configcond_config = {}

In [ ]:
cond_config = {}
for sp in usage_results:
    sp_data = usage_results[sp]
    keys = sp_data['idx_to_label'].values()
    vals = sp_data['idx_to_label'].keys()
    cond_config[sp] = dict(zip(keys, vals))


Combine per-site predictions and per-tissue correlations

In [ ]:
# Combine summary stats and original data for sll species
site_usage_df_corr = usage_df_all.merge(
    site_corr_all,
    on=["species", "tissue", "chr_pos"],
    how="left"
)

# Replace NaN in mean_true_usage with 0
site_usage_df_corr["mean_true_usage"] = site_usage_df_corr["mean_true_usage"].fillna(0)

# Split chr_pos -> chromosome, position
site_usage_df_corr[["chromosome", "position"]] = site_usage_df_corr["chr_pos"].str.split(":", expand=True)
site_usage_df_corr["position"] = site_usage_df_corr["position"].astype(int)

# Reorder columns: species, chromosome, position, then the rest
cols = ["species", "chromosome", "position"] + [c for c in site_usage_df_corr.columns if c not in ["species", "chromosome", "position"]]
site_usage_df_corr = site_usage_df_corr[cols]

site_usage_df_corr

#### Select top-correlated sites

Sort the data by top correlation

In [ ]:
metric = "mae"
# Calculate median cor per site (across tissues) for sorting
site_corr_all[f'median_{metric}'] = site_corr_all.groupby(['species', 'chr_pos'])[metric].transform('median')
# Sort sites by species and median correlation metric 
asc = True if metric in ["mae"] else False  # ascending for error metrics, descending for correlation metrics
sorted_sites = site_corr_all[['species', 'chr_pos', f'median_{metric}']].drop_duplicates().sort_values(['species', f'median_{metric}'], ascending=[True, asc]).reset_index(drop=True)
# Create id column for categories
sorted_sites['site_id'] = sorted_sites['species'] + "_" + sorted_sites['chr_pos']

In [ ]:
# How many site have high median correlation (e.g. median_pearson_r > 0.5)?
if metric.startswith("mae"):
    threshold = 0.4
    top_sites = sorted_sites[sorted_sites[f'median_{metric}'] < threshold]
    print(f"Number of sites with median {metric} < {threshold}: {len(top_sites)} / {len(sorted_sites)} ({len(top_sites) / len(sorted_sites):.2%})")
else:
    threshold = 0.6
    top_sites = sorted_sites[sorted_sites[f'median_{metric}'] > threshold]
    print(f"Number of sites with median {metric} > {threshold}: {len(top_sites)} / {len(sorted_sites)} ({len(top_sites) / len(sorted_sites):.2%})")

In [ ]:
# Order site_usage_df_corr by this sorted order of sites
site_usage_df_corr['site_id'] = site_usage_df_corr['species'] + "_" + site_usage_df_corr['chr_pos']
site_usage_df_corr['site_id'] = site_usage_df_corr['site_id'].astype(str)
site_usage_df_corr['site_id'] = pd.Categorical(site_usage_df_corr['site_id'], categories=sorted_sites['site_id'], ordered=True)
site_usage_df_corr = site_usage_df_corr.sort_values('site_id').reset_index(drop=True)

In [ ]:
# How many sites cdidn't have correlation metrics and thus have NaN site_id after merging?
print(f"Removing {site_usage_df_corr['site_id'].isna().sum()} rows corresponding to {site_usage_df_corr[site_usage_df_corr['site_id'].isna()][['species', 'chr_pos']].drop_duplicates().shape[0]} sites with missing site_id after merging.")
site_usage_df_corr = site_usage_df_corr[~site_usage_df_corr['site_id'].isna()].copy().reset_index(drop=True)

In [ ]:
# How many sites have NaN correlation (because all true values are 0)?
print(f"Number of sites with NaN correlation: {site_usage_df_corr['pearson_r'].isna().sum()} / {len(site_usage_df_corr)} ({site_usage_df_corr['pearson_r'].isna().mean():.2%})")

Select top examples to plot

In [ ]:
df = site_usage_df_corr[
    (site_usage_df_corr["site_id"].isin(top_sites['site_id']))
    & (site_usage_df_corr["n_obs_usage"] > 5)
]
df

examples_top = df[['species', 'chr_pos']].drop_duplicates().head(10)
example_coords = [f"{row['species']} {row['chr_pos']}" for _, row in examples_top.iterrows()]
print(f"Top {len(example_coords)} examples:", example_coords)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = os.path.join(work_dir, model, pd_key, f"splice_sites_trajectory_top_{metric}.pdf")

with PdfPages(pdf_path) as pdf:
    for coords in example_coords:
        fig = plot_splice_site_usage_by_tissue(
            df_all=site_usage_df_corr[site_usage_df_corr['true_usage'] > 0],
            site_coords=coords,
            data_config=cond_config,
            tissue_order=TISSUE_ORDER,
            tissue_colors=TISSUE_COLORS,
            figsize=(8, 6),
            verbose=False
        )
        if fig is not None:
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)  # free memory after saving each page

#### Select tissue-specific sites

In [ ]:
# Identify sites that are low in all tissues (mean_true_usage < 0.1) except selected tissues where they are high (mean_true_usage > 0.5)
# and check if these sites have higher correlation in the selected tissues

# Define thresholds
low_threshold = 0.2
high_threshold = 0.7

# Define selected tissues for each species
selected_tissues_by_species = {
    "human": ["Brain", "Cerebellum", "Midbrain"],
    "mouse": ["Brain", "Cerebellum"],
    "rat": ["Brain", "Cerebellum"],
    "rabbit": ["Brain", "Cerebellum"],
    "opossum": ["Brain", "Cerebellum"],
}
specific_sites = {}
for sp, tissues in selected_tissues_by_species.items():
    
    sp_data = site_usage_df_corr[site_usage_df_corr["species"] == sp]

    # Identify sites that are high in selected tissues
    tissue_data = sp_data[sp_data["tissue"].isin(tissues)]

    # Identify sites that are low in all tissues except selected ones
    other_tissues = [t for t in sp_data["tissue"].unique() if t not in tissues]
    other_data = sp_data[sp_data["tissue"].isin(other_tissues)]

    # Select sites that are low in other tissues and high in selected tissues
    low_in_others = other_data.groupby("chr_pos")["mean_true_usage"].max() < low_threshold
    print(f"{sp} - {tissues}: {low_in_others.sum()} sites that are low in other tissues")
    high_in_this = tissue_data.groupby("chr_pos")["mean_true_usage"].min() > high_threshold
    print(f"{sp} - {tissues}: {high_in_this.sum()} sites that are high in selected tissues")

    selected_low_sites = low_in_others.index[low_in_others]
    selected_high_sites = high_in_this.index[high_in_this]
    selected_sites = set(selected_low_sites) & set(selected_high_sites)
    print(f"{sp} - {tissues}: {len(selected_sites)} sites that are low in other tissues and high in selected tissues")

    # Check correlation for these sites in the selected tissue vs others
    corr_selected = tissue_data[tissue_data["chr_pos"].isin(selected_sites)]["pearson_r"].dropna()
    corr_others = other_data[other_data["chr_pos"].isin(selected_sites)]["pearson_r"].dropna()

    specific_sites[sp] = {
        "tissue": tissue,
        "n_selected_sites": len(selected_sites),
        "selected_sites": selected_sites,
    }

In [ ]:
# extract all sites from dictionary with structure e.g.
all_specific_sites = set()
for sp, info in specific_sites.items():
    all_specific_sites.update({f"{sp}_{site}" for site in info["selected_sites"]})
print(f"All specific sites: {len(all_specific_sites)}", all_specific_sites)

Select top examples to plot

In [ ]:
site_usage_df_corr

In [ ]:
df = site_usage_df_corr[
    (site_usage_df_corr["site_id"].isin(all_specific_sites))
    & (site_usage_df_corr["mae"] < 0.4)
    & (site_usage_df_corr["n_obs_usage"] > 6)
]
df = df.sort_values(['pearson_r'], ascending=False)

examples_top = df['site_id'].drop_duplicates().head(10)
example_coords = [site.replace("_", " ") for site in examples_top]
print(f"Top {len(example_coords)} examples:", example_coords)

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

pdf_path = os.path.join(work_dir, model, pd_key, "splice_sites_trajectory_tissue_specific.pdf")

with PdfPages(pdf_path) as pdf:
    for coords in example_coords:
        try:
            fig = plot_splice_site_usage_by_tissue(
                df_all=site_usage_df_corr, #[site_usage_df_corr['true_usage'] > 0],
                site_coords=coords,
                data_config=cond_config,
                tissue_order=TISSUE_ORDER,
                tissue_colors=TISSUE_COLORS,
                figsize=(8, 6),
                verbose=False
            )
            if fig is not None:
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)  # free memory after saving each page
        except Exception as e:
            print(f"Error plotting {coords}: {e}")

## Evaluation by developmental dynamics

For developmental dynamics, we need to lose inferred 0s for splice sites that are not observed in a given condition, since these are not necessarily true 0s but rather missing data.

In [ ]:
import copy

usage_results_filt = copy.deepcopy(usage_results)
for sps in usage_results_filt:
    if usage_results_filt[sps] is not None:
        mask = usage_results_filt[sps]['trues'] > 0
        for key in ['chr_pos', 'cond_ids', 'trues', 'preds']:
            usage_results_filt[sps][key] = usage_results_filt[sps][key][mask]
        for key in ['tissues', 'timepoints']:
            usage_results_filt[sps][key] = [usage_results_filt[sps][key][i] for i in range(len(usage_results_filt[sps][key])) if mask[i]]

usage_results_filt['human']['trues']


A site is classified as developmentally dynamic if the maximum amplitude of SSE change is greater than 0.2.

Dynamic sites are classified into four main developmental patterns:  
- Up: A progressive increase in SSE during development.
- Down: A progressive decrease in SSE during development.
- Up–Down: An initial increase in SSE followed by a decrease.
- Down–Up: An initial decrease in SSE followed by an increase.

To categorise these patterns, use a multi-step approach:  

- Use a cubic splines to approximate how SSE depends on developmental timepoint.
- Interpolate SSE values into 1,000 evenly distributed timepoints, and calculate the "SSE change" between each consecutive point.
- Derive four statistics for each site:
    - `up`: The sum of all positive SSE changes.
    - `down`: The absolute value of the sum of all negative SSE changes.
    - `up_timing`: The weighted average age of the positive changes (sum of positive changes multiplied by timepoint, then divided by the total `up` value).
    - `down_timing`: The weighted average age of the negative changes (absolute sum of negative changes multiplied by timepoint, then divided by the total `down` value).

Splice sites are assigned to one of the four classes based on the ratio of positive changes to the total change (`up`/(`up`+`down`)):  
- `down`: Assigned if the ratio is less than 0.3  
- `up`: Assigned if the ratio is greater than 0.7  
- `up–down`: For sites with a ratio between 0.3 and 0.7, if `up_timing`<`down_timing`  
- `down–up`: For sites with a ratio between 0.3 and 0.7, if `up_timing`>`down_timing`  

#### 1) Classify sites for each species and tissue

In [ ]:
from scipy.interpolate import make_interp_spline

dynamics_results = {}

for sps in SPECIES_ORDER:
    if sps not in usage_results_filt:
        continue
    sp_data = usage_results_filt[sps]
    if sp_data is None:
        continue

    tissues_available = sorted(set(sp_data['tissues']))
    dynamics_results[sps] = {}

    for tissue in tissues_available:
        df_cls = classify_sites_by_true_usage_variance(
            usage_data=sp_data,
            tissue=tissue,
        )
        dynamics_results[sps][tissue] = df_cls

#### 2) Inspect splice usage dynamics per species, tissue and subclass

In [ ]:
class_rows = []
subclass_rows = []

for sps, tissue_map in dynamics_results.items():
    if not tissue_map:
        continue

    for tissue, df_cls in tissue_map.items():
        if df_cls is None or df_cls.empty:
            continue

        # 1) Count unique sites per dynamics class
        class_counts = (
            df_cls.groupby("dynamics_class", as_index=False)["chr_pos"]
            .nunique()
            .rename(columns={"chr_pos": "n_sites"})
        )
        class_counts["species"] = sps
        class_counts["tissue"] = tissue
        class_rows.append(class_counts)

        # 2) Count unique sites per dynamics subclass
        subclass_counts = (
            df_cls.groupby("dynamics_subclass", as_index=False)["chr_pos"]
            .nunique()
            .rename(columns={"chr_pos": "n_sites"})
        )
        subclass_counts["species"] = sps
        subclass_counts["tissue"] = tissue
        subclass_rows.append(subclass_counts)

class_counts_df = pd.concat(class_rows, ignore_index=True) if class_rows else pd.DataFrame(
    columns=["species", "tissue", "dynamics_class", "n_sites"]
)
subclass_counts_df = pd.concat(subclass_rows, ignore_index=True) if subclass_rows else pd.DataFrame(
    columns=["species", "tissue", "dynamics_subclass", "n_sites"]
)

# Keep a single table name if needed by downstream cells
counts_df = subclass_counts_df.copy()

if class_counts_df.empty and subclass_counts_df.empty:
    print("No classified sites found in dynamics_results.")
else:
    if not class_counts_df.empty:
        class_counts_df["tissue"] = pd.Categorical(class_counts_df["tissue"], categories=TISSUE_ORDER, ordered=True)
        class_counts_df["dynamics_class"] = pd.Categorical(
            class_counts_df["dynamics_class"], categories=["stable", "dynamic"], ordered=True
        )
        class_counts_df = class_counts_df.sort_values(["species", "tissue", "dynamics_class"]).reset_index(drop=True)
        display(class_counts_df)

    if not subclass_counts_df.empty:
        subclass_counts_df["tissue"] = pd.Categorical(subclass_counts_df["tissue"], categories=TISSUE_ORDER, ordered=True)
        subclass_counts_df["dynamics_subclass"] = pd.Categorical(
            subclass_counts_df["dynamics_subclass"],
            categories=["high", "median", "low", "up", "down", "up-down", "down-up", "none"],
            ordered=True
        )
        subclass_counts_df = subclass_counts_df.sort_values(
            ["species", "tissue", "dynamics_subclass"]
        ).reset_index(drop=True)
        display(subclass_counts_df)

In [ ]:
# Toggle: False = raw lines (default), True = cubic spline smoothing
USE_SPLINES = False

# Helper: cubic spline plotting (falls back to straight line if not enough points)
def plot_cubic_spline(ax, x, y, color, linewidth, alpha=1.0, zorder=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # unique/sorted x for spline stability
    x_u, idx = np.unique(x, return_index=True)
    y_u = y[idx]
    order = np.argsort(x_u)
    x_u, y_u = x_u[order], y_u[order]

    if len(x_u) >= 4:  # cubic needs at least 4 points
        x_smooth = np.linspace(x_u.min(), x_u.max(), 200)
        y_smooth = make_interp_spline(x_u, y_u, k=3)(x_smooth)
        y_smooth = np.clip(y_smooth, 0, 1)
        ax.plot(x_smooth, y_smooth, color=color, linewidth=linewidth, alpha=alpha, zorder=zorder)
    else:
        ax.plot(x_u, y_u, color=color, linewidth=linewidth, alpha=alpha, zorder=zorder)

def plot_curve(ax, x, y, color, linewidth, alpha=1.0, zorder=None, use_splines=False):
    if use_splines:
        plot_cubic_spline(ax, x, y, color=color, linewidth=linewidth, alpha=alpha, zorder=zorder)
    else:
        ax.plot(x, y, color=color, linewidth=linewidth, alpha=alpha, zorder=zorder)

SUBCLASS_ORDER = ['up', 'down', 'up-down', 'down-up', 'high', 'median', 'low']

for sps in SPECIES_ORDER:
    if sps not in dynamics_results:
        continue

    tissues_available = sorted(dynamics_results[sps].keys())
    if not tissues_available:
        continue

    sp_data = usage_results_filt[sps]
    df_raw = pd.DataFrame({
        'chr_pos': sp_data['chr_pos'],
        'tissue': sp_data['tissues'],
        'timepoint': pd.to_numeric(sp_data['timepoints'], errors='coerce'),
        'true_usage': sp_data['trues'].astype(float),
    }).dropna(subset=['timepoint'])
    df_raw['timepoint'] = df_raw['timepoint'].astype(int)

    if df_raw.empty:
        continue

    tp_min = int(df_raw['timepoint'].min())
    tp_max = int(df_raw['timepoint'].max())
    x_ticks = np.arange(tp_min + 1, tp_max + 1, 2, dtype=int)

    n_tissues = len(tissues_available)
    n_subclasses = len(SUBCLASS_ORDER)

    fig, axes = plt.subplots(
        n_tissues, n_subclasses,
        figsize=(n_subclasses * 2.5, n_tissues * 2),
        sharex=True,
        sharey=True,
    )
    if n_tissues == 1:
        axes = axes[np.newaxis, :]
    if n_subclasses == 1:
        axes = axes[:, np.newaxis]

    for row, tissue in enumerate(tissues_available):
        df_cls = dynamics_results[sps][tissue]
        tissue_color = TISSUE_COLORS.get(tissue, '#444444')

        if df_cls.empty:
            for col in range(n_subclasses):
                ax = axes[row, col]
                ax.text(0.5, 0.5, 'n=0', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
            continue

        df_merged = df_raw[df_raw['tissue'] == tissue].merge(
            df_cls[['chr_pos', 'dynamics_subclass']], on='chr_pos', how='inner'
        )

        for col, subclass in enumerate(SUBCLASS_ORDER):
            ax = axes[row, col]
            df_sub = df_merged[df_merged['dynamics_subclass'] == subclass]
            n_sites = df_sub['chr_pos'].nunique()

            if df_sub.empty:
                ax.text(0.5, 0.5, 'n=0', ha='center', va='center',
                        transform=ax.transAxes, fontsize=8, color='gray')
            else:
                site_tp_mean = (
                    df_sub.groupby(['chr_pos', 'timepoint'])['true_usage']
                    .mean()
                    .reset_index()
                )

                # Individual site curves + points
                for site in site_tp_mean['chr_pos'].unique():
                    site_data = site_tp_mean[site_tp_mean['chr_pos'] == site].sort_values('timepoint')
                    plot_curve(
                        ax=ax,
                        x=site_data['timepoint'].values,
                        y=site_data['true_usage'].values,
                        color=tissue_color,
                        linewidth=1,
                        alpha=0.15,
                        use_splines=USE_SPLINES,
                    )
                    ax.plot(
                        site_data['timepoint'], site_data['true_usage'],
                        linestyle='None', marker='o', markersize=2,
                        color=tissue_color, alpha=0.15
                    )

                # Median curve + points (only for timepoints with majority coverage)
                n_total_sites = site_tp_mean['chr_pos'].nunique()
                tp_coverage = (
                    site_tp_mean.groupby('timepoint')['chr_pos']
                    .nunique()
                    .reset_index(name='n_sites_with_measurement')
                )
                valid_tps = tp_coverage.loc[
                    tp_coverage['n_sites_with_measurement'] > (n_total_sites / 2),
                    'timepoint'
                ]

                tp_stats = (
                    site_tp_mean[site_tp_mean['timepoint'].isin(valid_tps)]
                    .groupby('timepoint')['true_usage']
                    .median()
                    .reset_index()
                    .sort_values('timepoint')
                )
                plot_curve(
                    ax=ax,
                    x=tp_stats['timepoint'].values,
                    y=tp_stats['true_usage'].values,
                    color=tissue_color,
                    linewidth=2.5,
                    alpha=1.0,
                    zorder=10,
                    use_splines=USE_SPLINES,
                )
                ax.plot(
                    tp_stats['timepoint'], tp_stats['true_usage'],
                    linestyle='None', marker='o', markersize=4,
                    color=tissue_color, zorder=11, label='Median'
                )

                ax.text(
                    0.97, 0.03, f'n={n_sites}',
                    ha='right', va='bottom', transform=ax.transAxes,
                    fontsize=7, color='gray'
                )

            ax.grid(axis='y', linestyle='--', alpha=0.4)
            if row == 0:
                ax.set_title(subclass, fontsize=9)
            if col == 0:
                ax.set_ylabel(tissue, fontsize=9, color=tissue_color)
            if row == n_tissues - 1:
                ax.set_xlabel('Timepoint', fontsize=8)

    for ax in axes.flat:
        ax.set_xlim(tp_min, tp_max)
        ax.set_xticks(x_ticks)
        ax.set_ylim(0, 1)

    plt.tight_layout()

    out_path = os.path.join(work_dir, model, f"dynamics_true_usage_{sps}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out_path}")

#### 3) Evaluate usage prediction per species, tissue and subclass

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Build a long dataframe with true/pred usage + dynamics subclass labels
rows = []

for sps in SPECIES_ORDER:
    if sps not in usage_results or sps not in dynamics_results:
        continue
    sp_data = usage_results[sps]
    if sp_data is None:
        continue

    df_usage = pd.DataFrame({
        "species": sps,
        "chr_pos": np.asarray(sp_data["chr_pos"]).astype(str),
        "tissue": np.asarray(sp_data["tissues"]).astype(str),
        "timepoint": pd.to_numeric(np.asarray(sp_data["timepoints"]), errors="coerce"),
        "true_usage": np.asarray(sp_data["trues"], dtype=float),
        "pred_usage": np.asarray(sp_data["preds"], dtype=float),
    }).dropna(subset=["timepoint"])

    cls_frames = []
    for tissue, df_cls_t in dynamics_results[sps].items():
        if df_cls_t is None or df_cls_t.empty:
            continue
        cls_frames.append(
            df_cls_t[["chr_pos", "tissue", "dynamics_class", "dynamics_subclass"]].drop_duplicates()
        )

    if not cls_frames:
        continue

    df_cls_sp = pd.concat(cls_frames, ignore_index=True).drop_duplicates(["chr_pos", "tissue"])
    df_sp = df_usage.merge(df_cls_sp, on=["chr_pos", "tissue"], how="inner")
    rows.append(df_sp)

df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
    columns=[
        "species", "chr_pos", "tissue", "timepoint",
        "true_usage", "pred_usage", "dynamics_class", "dynamics_subclass"
    ]
)

def _pearson_safe(g):
    x = g["true_usage"].to_numpy(dtype=float)
    y = g["pred_usage"].to_numpy(dtype=float)
    if len(x) < 2 or np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return np.corrcoef(x, y)[0, 1]

def _mae_safe(g):
    x = g["true_usage"].to_numpy(dtype=float)
    y = g["pred_usage"].to_numpy(dtype=float)
    return mean_absolute_error(x, y) if len(x) > 0 else np.nan

def _rmse_safe(g):
    x = g["true_usage"].to_numpy(dtype=float)
    y = g["pred_usage"].to_numpy(dtype=float)
    return np.sqrt(mean_squared_error(x, y)) if len(x) > 0 else np.nan

corr_df = (
    df.groupby(["species", "tissue", "dynamics_subclass"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_obs": len(g),
          "n_sites": g["chr_pos"].nunique(),
          "pearson_r": _pearson_safe(g),
          "mae": _mae_safe(g),
          "rmse": _rmse_safe(g),
      }))
      .reset_index(drop=True)
)

corr_df["r_squared"] = corr_df["pearson_r"] ** 2
corr_df["n_obs"] = corr_df["n_obs"].astype(int)
corr_df["n_sites"] = corr_df["n_sites"].astype(int)
corr_df = corr_df.sort_values(["species", "tissue", "dynamics_subclass"]).reset_index(drop=True)

display(corr_df)

In [ ]:
# Plot total n_sites per dynamics subclass (stacked by species)

if "subclass_counts_df" in globals() and not subclass_counts_df.empty:
    plot_df = subclass_counts_df.copy()
elif "corr_df" in globals() and not corr_df.empty:
    plot_df = corr_df[["species", "dynamics_subclass", "n_sites"]].copy()
else:
    raise ValueError("No suitable dataframe found (expected subclass_counts_df or corr_df).")

summary_df = (
    plot_df.groupby(["dynamics_subclass", "species"], as_index=False)["n_sites"]
    .sum()
)

subclass_order = [s for s in SUBCLASS_ORDER if s in summary_df["dynamics_subclass"].unique()]
species_order = [s for s in SPECIES_ORDER if s in summary_df["species"].unique()]

pivot_df = (
    summary_df.pivot(index="dynamics_subclass", columns="species", values="n_sites")
    .reindex(subclass_order)
    .fillna(0)
)

ax = pivot_df.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    width=0.8,
    colormap="tab20"
)

ax.set_title("Number of sites per dynamics subclass")
ax.set_xlabel("Dynamics subclass")
ax.set_ylabel("n_sites")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(title="Species", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
species_plot = [s for s in SPECIES_ORDER if s in corr_df["species"].unique()]
subclass_plot = [s for s in SUBCLASS_ORDER if s in corr_df["dynamics_subclass"].unique()]
tissue_plot = [t for t in TISSUE_ORDER if t in corr_df["tissue"].unique()]

fig, axes = plt.subplots(
    len(species_plot), 1,
    figsize=(max(len(subclass_plot), 6), max(7 * len(species_plot), 6)),
    sharex=False
)

if len(species_plot) == 1:
    axes = [axes]

for i, (ax, sp) in enumerate(zip(axes, species_plot)):
    df_sp = corr_df[corr_df["species"] == sp]

    mat_r = (
        df_sp.pivot_table(
            index="tissue",
            columns="dynamics_subclass",
            values="pearson_r",
            aggfunc="mean"
        )
        .reindex(index=tissue_plot, columns=subclass_plot)
    )

    mat_n = (
        df_sp.pivot_table(
            index="tissue",
            columns="dynamics_subclass",
            values="n_sites",
            aggfunc="sum"
        )
        .reindex(index=tissue_plot, columns=subclass_plot)
    )

    annot = mat_r.copy().astype(object)
    for r in mat_r.index:
        for c in mat_r.columns:
            rv = mat_r.loc[r, c]
            nv = mat_n.loc[r, c]
            if pd.isna(rv):
                annot.loc[r, c] = ""
            else:
                n_txt = "NA" if pd.isna(nv) else f"{int(nv)}"
                annot.loc[r, c] = f"{rv:.2f}\n(n={n_txt})"

    sns.heatmap(
        mat_r,
        ax=ax,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        center=0,
        annot=annot,
        fmt="",
        linewidths=0.4,
        linecolor="white"
    )

    ax.set_title(sp)
    ax.set_xlabel("Dynamics subclass")
    ax.set_ylabel("Tissue")

plt.tight_layout()
plt.show()

### Plot site dynamics by developmental timepoint